# Raw Dataset Processor

Pipeline monitoring untuk mengubah raw dataset `datasets/url_discovery/*.json` menjadi kandidat dataset dengan schema seperti `datasets/v1_shs_datasets.csv`. Notebook ini tidak menyimpan output secara otomatis.

## 1. Setup

In [1]:
from pathlib import Path
import sys

import polars as pl
from IPython.display import display


def find_project_root(start: Path) -> Path:
    current = start.resolve()
    for candidate in (current, *current.parents):
        if (candidate / "config.py").exists() and (candidate / "services").is_dir():
            return candidate
    raise FileNotFoundError("Root proyek tidak ditemukan")


PROJECT_ROOT = find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import config
from services.dataset_service import DatasetService

pl.Config.set_tbl_hide_column_data_types(True)
pl.Config.set_tbl_cell_alignment("LEFT")
pl.Config.set_fmt_str_lengths(250)
pl.Config.set_tbl_width_chars(180)
pl.Config.set_tbl_cols(-1)
pl.Config.set_tbl_rows(20)

dataset_service = DatasetService()
RAW_FOLDER = config.DATASETS / "url_discovery"
RESEARCH_CONFIG_PATH = PROJECT_ROOT / "research_config.json"
REFERENCE_DATASET_PATH = config.DATASETS / "reference_datasets.csv"

print(f"Raw folder: {RAW_FOLDER}")
print(f"Research config: {RESEARCH_CONFIG_PATH}")
print(f"Reference schema: {REFERENCE_DATASET_PATH}")

Raw folder: E:\School\tugas-akhir\project\datasets\url_discovery
Research config: E:\School\tugas-akhir\project\research_config.json
Reference schema: E:\School\tugas-akhir\project\datasets\reference_datasets.csv


## 2. Load Config dan Raw Discovery

In [2]:
research_config = dataset_service.load_research_config(RESEARCH_CONFIG_PATH)
meta_df = dataset_service.load_url_discovery_meta(RAW_FOLDER)
queries_df = dataset_service.load_url_discovery_queries(RAW_FOLDER)
raw_records_df = dataset_service.load_url_discovery_records(RAW_FOLDER)

RECORDS_FILTER_CONDITIONS = [
    # pl.col("content_status") == "success",
    # pl.col("content_text").fill_null("").str.strip_chars().str.len_chars() > 0,
    # ~pl.col("domain").str.contains("instagram.com|tiktok.com|facebook.com"),
]


def apply_filter_conditions(df: pl.DataFrame, conditions: list) -> pl.DataFrame:
    if not conditions:
        return df
    expression = conditions[0]
    for condition in conditions[1:]:
        expression = expression & condition
    return df.filter(expression)


records_df = apply_filter_conditions(raw_records_df, RECORDS_FILTER_CONDITIONS)

print("Search label:", research_config.get("search_label"))
print("Primary location:", research_config.get("primary_location"))
print(f"File metadata: {meta_df.height:,}")
print(f"Query rows: {queries_df.height:,}")
print(f"Raw records unik: {raw_records_df.height:,}")
print(f"Records setelah filter manual: {records_df.height:,}")

display(meta_df)

Search label: SHS/PLTS Kalimantan Barat
Primary location: Kalimantan Barat
File metadata: 8
Query rows: 2,067
Raw records unik: 1,364
Records setelah filter manual: 1,364


source_file,built_at,search_label,source_files,record_count,content_count,query_count
"""dataset_20260627T094855Z.json""","""2026-06-27T09:48:55.240893+00:00""","""SHS/PLTS Kalimantan Barat""","{""E:\Work\personal\url-discovery\data\urls.json"",""E:\Work\personal\url-discovery\data\contents.json"",""E:\Work\personal\url-discovery\data\search_state.json""}",214,214,335
"""dataset_20260627T101253Z.json""","""2026-06-27T10:12:53.267116+00:00""","""SHS/PLTS Kalimantan Barat""","{""E:\Work\personal\url-discovery\data\urls.json"",""E:\Work\personal\url-discovery\data\contents.json"",""E:\Work\personal\url-discovery\data\search_state.json""}",203,203,268
"""dataset_20260627T135259Z.json""","""2026-06-27T13:52:59.725144+00:00""","""SHS/PLTS Kalimantan Barat""","{""E:\Work\personal\url-discovery\data\urls.json"",""E:\Work\personal\url-discovery\data\contents.json"",""E:\Work\personal\url-discovery\data\search_state.json""}",88,88,197
"""dataset_20260628T033631Z.json""","""2026-06-28T03:36:31.219965+00:00""","""SHS/PLTS Kalimantan Barat""","{""E:\Work\personal\url-discovery\data\urls.json"",""E:\Work\personal\url-discovery\data\contents.json"",""E:\Work\personal\url-discovery\data\search_state.json""}",202,202,381
"""dataset_20260628T104512Z.json""","""2026-06-28T10:45:12.810124+00:00""","""SHS/PLTS Kalimantan Barat""","{""E:\Work\personal\url-discovery\data\urls.json"",""E:\Work\personal\url-discovery\data\contents.json"",""E:\Work\personal\url-discovery\data\search_state.json""}",136,136,369
"""dataset_20260628T113525Z.json""","""2026-06-28T11:35:25.172805+00:00""","""SHS/PLTS Kalimantan Barat""","{""E:\Work\personal\url-discovery\data\urls.json"",""E:\Work\personal\url-discovery\data\contents.json"",""E:\Work\personal\url-discovery\data\search_state.json""}",12,12,28
"""dataset_20260628T173013Z.json""","""2026-06-28T17:30:13.810625+00:00""","""SHS/PLTS Kalimantan Barat""","{""E:\Work\personal\url-discovery\data\urls.json"",""E:\Work\personal\url-discovery\data\contents.json"",""E:\Work\personal\url-discovery\data\search_state.json""}",250,250,267
"""dataset_20260630T005918Z.json""","""2026-06-30T00:59:18.792990+00:00""","""SHS/PLTS Kalimantan Barat""","{""E:\Work\personal\url-discovery\data\urls.json"",""E:\Work\personal\url-discovery\data\contents.json"",""E:\Work\personal\url-discovery\data\search_state.json""}",259,259,490


## 3. Monitoring Kualitas Raw Dataset

In [3]:
status_distribution = (
    records_df.with_columns(pl.col("content_status").fill_null("missing"))
    .group_by("content_status")
    .agg(pl.len().alias("jumlah"))
    .sort("jumlah", descending=True)
)

domain_distribution = (
    records_df.with_columns(pl.col("domain").fill_null("unknown"))
    .group_by("domain")
    .agg(pl.len().alias("jumlah"))
    .sort("jumlah", descending=True)
    .head(20)
)

query_group_distribution = (
    records_df.with_columns(pl.col("query_group").fill_null("unknown"))
    .group_by("query_group")
    .agg(pl.len().alias("jumlah"))
    .sort("jumlah", descending=True)
    .head(20)
)

text_length_summary = records_df.select(
    pl.len().alias("total_records"),
    (
        pl.col("content_text")
        .fill_null("")
        .str.strip_chars()
        .str.len_chars()
        > 0
    ).sum().alias("records_with_text"),
    pl.col("content_text_length").min().alias("min_text_length"),
    pl.col("content_text_length").mean().round(2).alias("avg_text_length"),
    pl.col("content_text_length").max().alias("max_text_length"),
)

display(status_distribution)
display(domain_distribution)
display(query_group_distribution)
display(text_length_summary)

content_status,jumlah
"""success""",1067
"""too_short""",200
"""pdf_skipped""",47
"""failed""",41
"""pending""",9


domain,jumlah
"""www.instagram.com""",530
"""www.tiktok.com""",190
"""www.facebook.com""",56
"""pontianakpost.jawapos.com""",32
"""rri.co.id""",28
"""kalbar.antaranews.com""",25
"""www.researchgate.net""",12
"""id.scribd.com""",11
"""krisnamandiri.com""",10
"""www.krisnamandiriutama.com""",9


query_group,jumlah
"""core:electricity_access""",232
"""core:grid_hybrid""",115
"""core:household_solar""",103
"""issue:benefit""",93
"""issue:problem""",84
"""social""",75
"""core:technical""",68
"""core:government_program""",67
"""issue:access""",62
"""issue:environment""",55


total_records,records_with_text,min_text_length,avg_text_length,max_text_length
1364,1099,0,3257.82,33760


## 4. Bangun Kandidat Schema v1

In [4]:
candidate_df = dataset_service.build_v1_candidate_rows(
    records_df=records_df,
    research_config=research_config,
)

CANDIDATE_FILTER_CONDITIONS = [
    # pl.col("dataset_tier") == "C_holdout_excluded",
    # pl.col("source_type") == "social_media",
    # pl.col("source_url").str.to_lowercase().str.contains("facebook.com"),
    # ~pl.col("text").str.to_lowercase().str.contains("penayangan"),
    # pl.col("aspect").is_in(["experience", "benefit", "access"]),
]

filtered_candidate_df = apply_filter_conditions(
    candidate_df, CANDIDATE_FILTER_CONDITIONS
)

reference_df = dataset_service.load(REFERENCE_DATASET_PATH)
expected_columns = list(dataset_service.v1_dataset_columns())
missing_candidate_columns = [
    column for column in expected_columns if column not in filtered_candidate_df.columns
]
extra_candidate_columns = [
    column for column in filtered_candidate_df.columns if column not in expected_columns
]

## 5. Dataset Kandidat Siap Labeling


In [5]:
MAX_ROWS_PER_VALUE = 10

labeling_df = dataset_service.build_candidate_labeling_dataset(
    filtered_candidate_df,
    max_rows_per_value=MAX_ROWS_PER_VALUE,
)

candidate_combination_df = dataset_service.summarize_candidate_combinations(
    filtered_candidate_df
).rename({"jumlah": "jumlah_tersedia"})

valid_location_df = filtered_candidate_df.filter(
    (pl.col("is_specific_location") == True)
    & pl.col("location_source").is_in(["text", "query"])
)
valid_location_combination_df = dataset_service.summarize_candidate_combinations(
    valid_location_df
).rename({"jumlah": "jumlah_lokasi_valid"})

labeling_combination_df = dataset_service.summarize_labeling_selection_buckets(
    labeling_df
).rename({"jumlah": "jumlah_diambil"})

combination_audit_df = (
    candidate_combination_df
    .join(
        valid_location_combination_df,
        on=["combination_column", "combination_value"],
        how="left",
    )
    .join(
        labeling_combination_df,
        on=["combination_column", "combination_value"],
        how="left",
    )
    .with_columns(
        pl.col("jumlah_lokasi_valid").fill_null(0).cast(pl.Int64),
        pl.col("jumlah_diambil").fill_null(0).cast(pl.Int64),
    )
)

print(f"Candidate rows setelah filter kandidat: {filtered_candidate_df.height:,}")
print(f"Candidate rows dengan lokasi valid: {valid_location_df.height:,}")
print(f"Final siap labeling: {labeling_df.height:,}")
print(f"Maksimal data per nilai unik: {MAX_ROWS_PER_VALUE}")

display(combination_audit_df)


Candidate rows setelah filter kandidat: 1,364
Candidate rows dengan lokasi valid: 1,100
Final siap labeling: 164
Maksimal data per nilai unik: 10


combination_column,combination_value,jumlah_tersedia,jumlah_lokasi_valid,jumlah_diambil
"""source_type""","""academic_repository""",75,59,10
"""source_type""","""government_official""",46,30,10
"""source_type""","""online_news""",459,362,10
"""source_type""","""social_media""",784,649,10
"""aspect""","""access""",10,10,10
"""aspect""","""benefit""",106,82,10
"""aspect""","""electricity_access""",79,71,10
"""aspect""","""environment""",70,58,10
"""aspect""","""experience""",629,500,10
"""aspect""","""general_shs""",385,313,10


## 6. Preview Lokasi Valid


In [6]:
valid_location_columns = [
    "text_id",
    "location",
    "location_source",
    "location_match",
    "source_type",
    "aspect",
    "dataset_tier",
    "source_url",
    "query",
    "text",
]
valid_location_columns = [
    column for column in valid_location_columns if column in valid_location_df.columns
]

display(valid_location_df.select(valid_location_columns).head(10))

text_id,location,location_source,location_match,source_type,aspect,dataset_tier,source_url,query,text
"""RAW-0001""","""Singkawang""","""text""","""singkawang""","""online_news""","""experience""","""A_candidate_core""","""https://www.suarapemredkalbar.com/read/nasional/26062026/program-listrik-desa-di-papua-hadirkan-terang-dan-harapan-hingga-pelosok-negeri""","""""elektrifikasi"" ""penerangan desa"" ""Melawi"" after:2024-01-01 before:2026-06-27""","""Program Listrik Desa di Papua Hadirkan Terang dan Harapan Hingga Pelosok Negeri Program Listrik Desa di Papua Hadirkan Terang dan Harapan ... suarapemredkalbar.com https://www.suarapemredkalbar.com › read › nasional 26062026 program listrik desa di p…"
"""RAW-0002""","""Sambas""","""query""","""sambas""","""social_media""","""general_shs""","""B_review_queue""","""https://www.instagram.com/reel/DaCiaOBxC_d""","""""PLTS"" ""hemat tagihan listrik"" ""Sambas"" after:2024-01-01 before:2026-06-27""","""Ari Yunianto on Instagram: ""Bekasi beres. Semangat…!! Antrian project installasi PLTS masih full sampai Agustus #plts #panelsurya #offgrid #pltsatap #solarpanels"" Bekasi beres. Semangat…!! Antrian project installasi PLTS ... Instagram · ariyuniant0 2…"
"""RAW-0003""","""Sambas""","""query""","""sambas""","""social_media""","""general_shs""","""B_review_queue""","""https://www.instagram.com/reel/DY0bixATBPD""","""""PLTS"" ""hemat tagihan listrik"" ""Sambas"" after:2024-01-01 before:2026-06-27""","""OIS POWER on Instagram: ""#PLTS #PanelSurya #SolarPanel #EnergiTerbarukan #SolarEnergy"" OIS POWER | #PLTS #PanelSurya #SolarPanel ... Instagram · ois_power 6 suka · 1 bulan yang lalu 6 likes, 0 comments - ois_power on May 26, 2026: ""#PLTS #PanelSurya …"
"""RAW-0004""","""Sambas""","""query""","""sambas""","""social_media""","""general_shs""","""B_review_queue""","""https://www.instagram.com/p/DRq794LgAB2""","""""PLTS"" ""hemat tagihan listrik"" ""Sambas"" after:2024-01-01 before:2026-06-27""","""TEKNIK LISTRIK on Instagram: ""Panel Surya Listrik (Fotovoltaik/PV) adalah teknologi yang mengambil energi dari cahaya matahari (foton), bukan panas, lalu mengubahnya menjadi listrik DC yang kemudian diubah oleh Inverter menjadi listrik AC yang dapat …"
"""RAW-0005""","""Sambas""","""query""","""sambas""","""social_media""","""general_shs""","""B_review_queue""","""https://www.instagram.com/reel/DQlfNxOATsq""","""""PLTS"" ""hemat tagihan listrik"" ""Sambas"" after:2024-01-01 before:2026-06-27""","""Datascrip Service Center on Instagram: ""Waktu yg tepat untuk membersihkan panel surya."" Waktu yg tepat untuk membersihkan panel surya. Instagram · datascrip.service.center.id 1 suka · 7 bulan yang lalu 1 likes, 0 comments - datascrip.service.center.i…"
"""RAW-0006""","""Sambas""","""query""","""sambas""","""social_media""","""general_shs""","""C_holdout_excluded""","""https://www.tiktok.com/@evanstoryyy/video/7643019382251080967""","""""PLTS"" ""hemat tagihan listrik"" ""Sambas"" after:2024-01-01 before:2026-06-27""","""TikTok - Make Your Day Listrik Padam Sumatera: Solusi PLTS dan Stok Cadangan ... TikTok · evanstoryy 7,2 rb+ penayangan @evanstoryyy 7643019382251080967"""
"""RAW-0007""","""Sambas""","""query""","""sambas""","""social_media""","""general_shs""","""B_review_queue""","""https://www.instagram.com/p/DP2n1gQD6J_""","""""tenaga surya"" ""listrik 24 jam"" ""Sambas"" after:2024-01-01 before:2026-06-27""","""Damai Cable Indonesia on Instagram: ""Butuh kabel panel surya? Kami siap penuhi kebutuhan proyek energi terbarukanmu! Kuat, awet, dan sudah teruji kualitasnya. #damaicable"" Butuh kabel panel surya? Kami siap penuhi kebutuhan ... Instagram · damaicable…"
"""RAW-0008""","""Sambas""","""query""","""sambas""","""social_media""","""general_shs""","""C_holdout_excluded""","""https://www.tiktok.com/@bacaaja.co/video/7651132761448647956""","""""tenaga surya"" ""listrik 24 jam"" ""Sambas"" after:2024-01-01 before:2026-06-27""","""TikTok - Make Your Day Teknologi CSP China di Gurun Gobi: Pembangkit 

## 7. Final Dataset Siap Labeling


In [7]:
target_columns = list(dataset_service.v1_dataset_columns())
monitoring_columns = [
    "location_source",
    "location_match",
    "is_specific_location",
    "raw_source_file",
    "raw_domain",
    "content_status",
    "query_group",
    "query",
    "raw_title",
    "raw_text_length",
]
final_labeling_columns = [
    column for column in [*target_columns, *monitoring_columns]
    if column in labeling_df.columns
]

missing_final_columns = [
    column for column in target_columns if column not in labeling_df.columns
]
extra_final_columns = [
    column for column in labeling_df.columns if column not in target_columns
]

print("Kolom target hilang:", missing_final_columns)
print("Kolom monitoring tambahan:", extra_final_columns)
print(
    "Urutan kolom target sama:",
    labeling_df.columns[: len(target_columns)] == target_columns,
)

display(labeling_df.select(final_labeling_columns))


Kolom target hilang: []
Kolom monitoring tambahan: ['raw_source_file', 'raw_domain', 'content_status', 'query_group', 'query', 'raw_title', 'raw_text_length', 'location_source', 'location_match', 'is_specific_location', 'labeling_bucket_column', 'labeling_bucket_value']
Urutan kolom target sama: True


text_id,text,subjectivity_type,speaker_type,public_opinion_scope,corpus_role,aspect,location,sentiment_label,label_status,source_id,source_type,source_url,dataset_tier,inclusion_status,verification_status,evidence_support_score,parent_text_id,decision_note,location_source,location_match,is_specific_location,raw_source_file,raw_domain,content_status,query_group,query,raw_title,raw_text_length
"""RAW-0715""","""Analisis Perbandingan Instalasi Panel Surya Sistem On-Grid ... Jurnal Universitas Peradaban https://journal.peradaban.ac.id › article › download 2079 1288""","""contextual_source""","""researcher""","""contextual_reference""","""excluded""","""technical""","""Pontianak""","""""","""unlabeled""","""RAW-SRC-0715""","""academic_repository""","""https://journal.peradaban.ac.id/index.php/jeepa/article/download/2079/1288""","""C_holdout_excluded""","""held_out_not_for_sentiment_core""","""perlu_verifikasi""",0.65,"""""","""content_not_success""","""query""","""pontianak""",true,"""dataset_20260628T104512Z.json""","""journal.peradaban.ac.id""","""failed""","""pdf""","""filetype:pdf ""jaringan listrik PLN"" ""Pontianak"" after:2024-01-01 before:2026-06-27""","""Analisis Perbandingan Instalasi Panel Surya Sistem On-Grid ...""",0
"""RAW-0226""","""Zona Energi Baru Terbarukan on Instagram: ""Pada 8 Mei 2026, ZONAEBT sukses menyelenggarakan Program SINAR (Sinergi Energi Terbarukan untuk Rakyat) secara hybrid yang dilaksanakan di Gedung Teknik Elektro Politeknik Negeri Bali serta melalui Zoom 🌞🔋 D…","""public_expectation""","""public_user""","""public_opinion""","""contextual_evidence""","""village_solar""","""Kayong Utara""","""""","""unlabeled""","""RAW-SRC-0226""","""social_media""","""https://www.instagram.com/reel/DYbki9lvgnh""","""B_review_queue""","""review_required_before_core""","""perlu_verifikasi""",0.9,"""""","""social_domain_review""","""query""","""sukadana""",true,"""dataset_20260627T101253Z.json""","""www.instagram.com""","""success""","""issue:benefit""","""""energi terbarukan"" ""hemat tagihan listrik"" ""Sukadana"" after:2024-01-01 before:2026-06-27""","""Pada 8 Mei 2026, ZONAEBT sukses menyelenggarakan ...""",1228
"""RAW-0372""","""TikTok - Make Your Day Pengembangan PLTS Terpusat di Batoq Kelo TikTok · Dr H Bambang Arwanto AP MSi 14,9 rb+ penayangan · 5 bulan yang lalu @doktor.barwanto 7592448239425834261""","""contextual_source""","""public_user""","""contextual_reference""","""excluded""","""village_solar""","""Kapuas Hulu""","""""","""unlabeled""","""RAW-SRC-0372""","""social_media""","""https://www.tiktok.com/@doktor.barwanto/video/7592448239425834261""","""C_holdout_excluded""","""held_out_not_for_sentiment_core""","""perlu_verifikasi""",0.55,"""""","""content_not_success; social_domain_review""","""query""","""kapuas hulu""",true,"""dataset_20260627T101253Z.json""","""www.tiktok.com""","""too_short""","""issue:access""","""""tenaga surya"" ""daerah 3T"" ""Kapuas Hulu"" after:2024-01-01 before:2026-06-27""","""Pengembangan PLTS Terpusat di Batoq Kelo""",0
"""RAW-1253""","""TikTok - Make Your Day PLTS mini: kunci penting ing urip (PLTS off-grid) TikTok · PLTS MINI 2,2 rb+ penayangan · 3 minggu yang lalu @story sound.mini 7648252197137845524""","""contextual_source""","""public_user""","""contextual_reference""","""excluded""","""village_solar""","""Kubu Raya""","""""","""unlabeled""","""RAW-SRC-1253""","""social_media""","""https://www.tiktok.com/@story_sound.mini/video/7648252197137845524""","""C_holdout_excluded""","""held_out_not_for_sentiment_core""","""perlu_verifikasi""",0.55,"""""","""content_not_success; social_domain_review""","""query""","""kubu raya""",true,"""dataset_20260630T005918Z.json""","""www.tiktok.com""","""too_short""","""core:household_solar""","""""PLTS rumah"" ""Kubu Raya"" after:2024-01-01 before:2026-06-27""","""PLTS mini: kunci penting ing urip (PLTS off-grid)""",0
"""RAW-0560""","""Simulasi Pola Operasi Optimal PLTS-PLTD Pada Daerah 3T Menggunakan Metoda Hybrid AI (Studi Kasus: Sist

In [8]:
tier_distribution = (
    filtered_candidate_df.group_by("dataset_tier")
    .agg(pl.len().alias("jumlah"))
    .sort("jumlah", descending=True)
)

inclusion_distribution = (
    filtered_candidate_df.group_by("inclusion_status")
    .agg(pl.len().alias("jumlah"))
    .sort("jumlah", descending=True)
)

source_type_distribution = (
    filtered_candidate_df.group_by("source_type")
    .agg(pl.len().alias("jumlah"))
    .sort("jumlah", descending=True)
)

aspect_distribution = (
    filtered_candidate_df.group_by("aspect")
    .agg(pl.len().alias("jumlah"))
    .sort("jumlah", descending=True)
    .head(20)
)


location_distribution = (
    filtered_candidate_df.group_by("location")
    .agg(pl.len().alias("jumlah"))
    .sort("jumlah", descending=True)
    .head(20)
)

display(tier_distribution)
display(inclusion_distribution)
display(source_type_distribution)
display(aspect_distribution)
display(location_distribution)

dataset_tier,jumlah
"""B_review_queue""",581
"""A_candidate_core""",486
"""C_holdout_excluded""",297


inclusion_status,jumlah
"""review_required_before_core""",581
"""candidate_analysis_ready""",486
"""held_out_not_for_sentiment_core""",297


source_type,jumlah
"""social_media""",784
"""online_news""",459
"""academic_repository""",75
"""government_official""",46


aspect,jumlah
"""experience""",629
"""general_shs""",385
"""benefit""",106
"""electricity_access""",79
"""environment""",70
"""grid_hybrid""",49
"""problem""",18
"""household_solar""",13
"""access""",10
"""village_solar""",4


location,jumlah
"""Kalimantan Barat""",216
"""Pontianak""",129
"""Kubu Raya""",80
"""Ketapang""",78
"""Sambas""",73
"""Sintang""",73
"""Kapuas Hulu""",71
"""Bengkayang""",65
"""Sanggau""",59
"""Kayong Utara""",58


In [9]:
print(f"Candidate rows sebelum filter kandidat: {candidate_df.height:,}")
print(f"Candidate rows setelah filter kandidat: {filtered_candidate_df.height:,}")
print("Kolom wajib hilang:", missing_candidate_columns)
print("Kolom monitoring tambahan:", extra_candidate_columns)
print(
    "Urutan kolom utama sama:",
    filtered_candidate_df.columns[: len(expected_columns)] == expected_columns,
)

display(filtered_candidate_df.select(expected_columns).head(20))

Candidate rows sebelum filter kandidat: 1,364
Candidate rows setelah filter kandidat: 1,364
Kolom wajib hilang: []
Kolom monitoring tambahan: ['raw_source_file', 'raw_domain', 'content_status', 'query_group', 'query', 'raw_title', 'raw_text_length', 'location_source', 'location_match', 'is_specific_location']
Urutan kolom utama sama: True


text_id,text,subjectivity_type,speaker_type,public_opinion_scope,corpus_role,aspect,location,sentiment_label,label_status,source_id,source_type,source_url,dataset_tier,inclusion_status,verification_status,evidence_support_score,parent_text_id,decision_note
"""RAW-0001""","""Program Listrik Desa di Papua Hadirkan Terang dan Harapan Hingga Pelosok Negeri Program Listrik Desa di Papua Hadirkan Terang dan Harapan ... suarapemredkalbar.com https://www.suarapemredkalbar.com › read › nasional 26062026 program listrik desa di p…","""public_expectation""","""community_representative""","""public_opinion""","""core_public_opinion""","""experience""","""Singkawang""","""""","""unlabeled""","""RAW-SRC-0001""","""online_news""","""https://www.suarapemredkalbar.com/read/nasional/26062026/program-listrik-desa-di-papua-hadirkan-terang-dan-harapan-hingga-pelosok-negeri""","""A_candidate_core""","""candidate_analysis_ready""","""perlu_verifikasi""",1.0,"""""","""candidate_from_raw_discovery"""
"""RAW-0002""","""Ari Yunianto on Instagram: ""Bekasi beres. Semangat…!! Antrian project installasi PLTS masih full sampai Agustus #plts #panelsurya #offgrid #pltsatap #solarpanels"" Bekasi beres. Semangat…!! Antrian project installasi PLTS ... Instagram · ariyuniant0 2…","""public_experience""","""public_user""","""public_opinion""","""contextual_evidence""","""general_shs""","""Sambas""","""""","""unlabeled""","""RAW-SRC-0002""","""social_media""","""https://www.instagram.com/reel/DaCiaOBxC_d""","""B_review_queue""","""review_required_before_core""","""perlu_verifikasi""",0.9,"""""","""social_domain_review"""
"""RAW-0003""","""OIS POWER on Instagram: ""#PLTS #PanelSurya #SolarPanel #EnergiTerbarukan #SolarEnergy"" OIS POWER | #PLTS #PanelSurya #SolarPanel ... Instagram · ois_power 6 suka · 1 bulan yang lalu 6 likes, 0 comments - ois_power on May 26, 2026: ""#PLTS #PanelSurya …","""public_experience""","""public_user""","""public_opinion""","""contextual_evidence""","""general_shs""","""Sambas""","""""","""unlabeled""","""RAW-SRC-0003""","""social_media""","""https://www.instagram.com/reel/DY0bixATBPD""","""B_review_queue""","""review_required_before_core""","""perlu_verifikasi""",0.9,"""""","""social_domain_review"""
"""RAW-0004""","""TEKNIK LISTRIK on Instagram: ""Panel Surya Listrik (Fotovoltaik/PV) adalah teknologi yang mengambil energi dari cahaya matahari (foton), bukan panas, lalu mengubahnya menjadi listrik DC yang kemudian diubah oleh Inverter menjadi listrik AC yang dapat …","""public_experience""","""public_user""","""public_opinion""","""contextual_evidence""","""general_shs""","""Sambas""","""""","""unlabeled""","""RAW-SRC-0004""","""social_media""","""https://www.instagram.com/p/DRq794LgAB2""","""B_review_queue""","""review_required_before_core""","""perlu_verifikasi""",0.9,"""""","""social_domain_review"""
"""RAW-0005""","""Datascrip Service Center on Instagram: ""Waktu yg tepat untuk membersihkan panel surya."" Waktu yg tepat untuk membersihkan panel surya. Instagram · datascrip.service.center.id 1 suka · 7 bulan yang lalu 1 likes, 0 comments - datascrip.service.center.i…","""public_experience""","""public_user""","""public_opinion""","""contextual_evidence""","""general_shs""","""Sambas""","""""","""unlabeled""","""RAW-SRC-0005""","""social_media""","""https://www.instagram.com/reel/DQlfNxOATsq""","""B_review_queue""","""review_required_before_core""","""perlu_verifikasi""",0.9,"""""","""social_domain_review"""
"""RAW-0006""","""TikTok - Make Your Day Listrik Padam Sumatera: Solusi PLTS dan Stok Cadangan ... TikTok · evanstoryy 7,2 rb+ penayangan @evanstoryyy 7643019382251080967""","""contextual_source""","""public_user""","""contextual_reference""","""excluded""","""general_shs""","""Sambas""","""""","""unlabeled""","""RAW-SRC-0006""","""social_media""","""https://www.tiktok.com/@evanstoryyy/video/7643019382251080967""","""C_holdout_excluded""","""held_out_not_for_sentiment_core""","""perlu_verifikasi""",0.55,"""""","""content_not_succe